In [1]:
import batting_statlines as stat 
import pandas as pd 
import numpy as np 
import mysql.connector as sql
from IPython import display
from collections import namedtuple

pd.options.display.max_columns = None

In [2]:
fangraphs_board = pd.read_csv('pitching_1947_2001.csv')
fangraphs_board.rename(columns={'season': 'yearID', 'team': 'teamID'}, inplace = True)
fangraphs_board.drop(['G', 'GS', 'HR/FB', 'xFIP', 'xFIP-', 'playerid', 'K/9', 'BB/9', 'K/9+', 'BB/9+'], axis=1, inplace=True)
fangraphs_board = stat.order_by(fangraphs_board, ['yearID', 'teamID'], True)
fangraphs_board = fangraphs_board[fangraphs_board.teamID != '- - -']

## Here We Go!

The first thing that I want to do, is to group everybody into teams. In order to do this analysis, I'm going to be looking at Johnson and Schilling's numbers together as a single aggregate. I think that would be the best, becuase I'm going to compare him to other top 2 pitchers on teams so we're really comparing "groups" of players rather than individual players.

### The Plan

What I plan on doing is first, create the aggregate values for each column, for the top two pitchers in a team each year. After I do that I want to rank each pair of staff aces for each statistic. From there, create an average score for how they rank in each statistic to find the most "dominant" pairing of pitchers. Then, look across each year and see who has the highest "dominance" score.

In [3]:
years = fangraphs_board.yearID.unique()
teams = fangraphs_board.teamID.unique()
dont_do = ['K%', 'BB%', 'name', 'teamID']
team_dict = namedtuple('team_avgs', 'col_name dict')
def make_avgs_dict(year, team):
    team_dict = {}
    team_df = fangraphs_board[(fangraphs_board.yearID == year) & (fangraphs_board.teamID == team)]
    team_df = stat.order_by(team_df, 'WAR', False)
    team_df = team_df.head(2)
    for col in team_df.columns:
        if col not in dont_do:
            team_dict[col] = team_df[col].mean()
    return team_dict

dicts = {}
for year in years:
    for team in teams:
        years_teams = fangraphs_board[fangraphs_board.yearID == year].teamID.unique()
        if team in years_teams:
            t = team_dict(col_name=f'{team}_{year}', dict=make_avgs_dict(year, team))
            dicts[t.col_name] = t.dict


In [6]:
top2_aggs = pd.DataFrame(data=dicts.values(), index=dicts.keys())
top2_aggs.reset_index(inplace=True)
top2_aggs.rename(columns={'index': 'team_year'}, inplace=True)
top2_aggs

,team_year,yearID,W,L,IP,WHIP,HR/9,BABIP,ERA,FIP,ERA-,FIP-,K/BB+,WHIP+,K%+,BB%+,WAR
0,Athletics_1947,1947.0,15.5,10.0,251.65,1.315,0.485,0.2500,3.015,3.780,81.5,100.0,91.5,94.0,97.0,105.5,2.90
1,Braves_1947,1947.0,21.0,11.0,277.60,1.215,0.555,0.2580,2.925,3.285,72.5,83.0,155.5,84.5,116.5,75.0,5.50
2,Browns_1947,1947.0,8.0,14.5,185.10,1.470,0.610,0.3035,4.695,3.480,120.5,87.5,143.0,105.0,116.0,81.5,3.20
3,Cardinals_1947,1947.0,15.0,6.5,196.05,1.350,0.430,0.2990,3.105,3.030,74.5,74.5,168.5,94.0,131.5,78.5,4.45
4,Cubs_1947,1947.0,10.5,15.0,195.00,1.390,0.640,0.2785,3.800,3.665,97.0,95.5,119.0,96.5,107.5,90.0,2.55
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1781,Mariners_2020,2020.0,2.0,1.0,17.20,0.740,2.040,0.1520,3.060,4.550,78.0,113.0,307.0,57.0,96.0,31.0,0.20
1782,Rockies_2020,2020.0,2.5,0.5,18.00,1.035,0.500,0.2540,2.270,2.495,47.5,56.5,137.5,83.5,104.5,76.0,0.65
1783,Diamondbacks_2020,2020.0,1.0,0.5,17.60,1.110,1.760,0.2680,2.550,4.260,61.0,102.5,335.5,89.5,106.5,66.0,0.20
1784,Nationals_2020,2020.0,0.5,0.5,12.60,1.120,1.080,0.3095,3.145,2.585,73.5,61.0,349.5,90.5,150.0,89.0,0.40


In [9]:
year_2001
rank_2001 = top2_aggs[top2_aggs.yearID == 2001].rank(method='first', axis=0, numeric_only=True)
rank_2001['composite'] = rank_2001.mean(axis=1)
rank_2001.head()

,yearID,W,L,IP,WHIP,HR/9,BABIP,ERA,FIP,ERA-,FIP-,K/BB+,WHIP+,K%+,BB%+,WAR,composite
1209,1.0,29.0,6.0,28.0,7.0,1.0,9.0,7.0,5.0,7.0,5.0,23.0,7.0,20.0,10.0,27.0,12.000
1210,2.0,18.0,20.0,25.0,4.0,4.0,8.0,2.0,3.0,2.0,3.0,28.0,4.0,23.0,4.0,28.0,11.125
1211,3.0,27.0,8.0,22.0,12.0,2.0,25.0,3.0,4.0,3.0,4.0,25.0,12.0,22.0,8.0,26.0,12.875
1212,4.0,23.0,1.0,16.0,10.0,12.0,6.0,11.0,7.0,12.0,7.0,24.0,10.0,26.0,21.0,22.0,13.250
1213,5.0,19.0,10.0,29.0,5.0,10.0,2.0,6.0,10.0,10.0,9.0,16.0,6.0,25.0,24.0,24.0,13.125
